<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# C18's Plan, Run

The run of C18's pre-registered plan, [C13-C24.md § C18](../docs/hypotheses/experiments/C13-C24.md#c18-spreading-activation-has-a-cliff),
built from the piloted harness in [c18_cliff_controls.ipynb](c18_cliff_controls.ipynb). The model, the variants, the
two stages, the margins, the six clauses and their levels are the plan's and are not restated as choices here; the
pre-registered expectations are in the journal of 2026-10-31, § C18 Unparked: the Cliff Defined, the Test Piloted.

**What changes from the pilot's harness**, all written before any variant was run:

- *Levels*: each clause's intervals at 1 - 0.05 / 6; clause (1)'s two intervals, the pure network's, at
  1 - 0.05 / 12. The pilot's `ALPHA = 0.01` is gone. The controls are read at both levels and must read as they must at
  each; if any misreads at either, the run is void and the variants are not judged.
- *Seeds*: fresh. Every draw has its own generator, seeded from `[20261103, stage, model, point, N, draw]` (stage 1
  locate, 2 test, 3 bootstrap, 4 the 1.1 D reading), so the numbers do not depend on how the work is split over
  processes. A replaced draw takes the next graph from the same generator.
- *The cosmetic nit from #193's review*: a model that never passes 1% prints "none" for its first grid degree past 1%.

**Where the plan is silent, filled before the run** (and written in the journal's pre-run note):

- *Priming* (clause 6) is measured on the 200 test draws of the pure network at its D / 2 at N = 10 000, the size the
  threshold is located at. One hop is the seed's out-neighbours; two hops are the units at shortest directed distance
  exactly two. Since a draw may have no unit two hops out, the ratio pools units over draws: the mean activation of all
  two-hop units over the 200 draws divided by that of all one-hop units, with a percentile bootstrap over draws
  (10 000 resamples) at 1 - 0.05 / 6.
- *The 1.1 D reading* is taken at N = 10 000 with 200 fresh draws, as the mean reached fraction there over that at 2 D;
  for every variant that reads cliff, and for the controls that read cliff, for calibration.
- *An undefined ratio*: where no unit is activated in any draw at N = 10 000 the ratio is 0 / 0. It is reported as
  undefined, and since an undefined interval lies wholly on neither side of a margin, the reading's rule makes that
  side neither finite, extensive, holding nor falling.

In [ ]:
import multiprocessing as mp
import os
import time
import warnings

import numpy as np

# nbconvert runs a notebook from its own directory; the figures live under the repository's root.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
# Forking a kernel that holds threads draws a DeprecationWarning; the workers touch only numpy and their own arguments.
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*fork.*")

SEED = 20261103
WORKERS = 3  # of the machine's four CPUs, one left for the other agents building beside this run
SIZES = [100, 1000, 10000]
GRID = np.arange(0.25, 12.01, 0.25)
LOCATE_DRAWS, TEST_DRAWS, BOOT = 50, 200, 10000
REACH_LEVEL = 0.01  # the located threshold D is where the mean reached fraction first passes 1%
LEVEL_CLAUSE = 1 - 0.05 / 6  # clauses (2) to (6), and the controls
LEVEL_PURE = 1 - 0.05 / 12  # clause (1)'s two intervals, and the controls again
MARGINS = {"finite below": 10.0, "extensive below": 3.0, "holding above": 2.0, "falling above": 5.0}
PRIMING_MARGIN = 0.5
SETTLE_TOL, SETTLE_STEPS, ACTIVE = 1e-10, 20000, 0.1
EDGE, DECAY, INHIBITION = 0.2, 0.5, 0.5
JUMP_AT, SMOOTH_MID, SMOOTH_WIDTH = 2.5, 5.0, 0.5

CONTROLS = ["giant component", "jump", "smooth", "neighbours"]
MUST_READ = {"giant component": "cliff", "jump": "cliff", "smooth": "no cliff", "neighbours": "no cliff"}
# Each variant is the pure network with one switch moved off its baseline; all switches at baseline is the pure network.
BASELINE = {"budget": None, "fanout": False, "inhibition": False}
VARIANTS = {
    "pure": {},
    "budget c = 0.2": {"budget": 0.2},
    "budget c = 0.6": {"budget": 0.6},
    "fan-out": {"fanout": True},
    "inhibition": {"inhibition": True},
}
KINDS = CONTROLS + list(VARIANTS)
CLAUSE = {"pure": 1, "inhibition": 2, "budget c = 0.2": 3, "budget c = 0.6": 4, "fan-out": 5}


def network(n, d, rng):
    """The pilot's directed random graph: Poisson(d n) edges with uniform ends, self-loops dropped, repeats kept."""
    m = rng.poisson(d * n)
    src, dst = rng.integers(0, n, m), rng.integers(0, n, m)
    keep = src != dst
    return src[keep], dst[keep]


def out_component(n, src, dst, seed=0):
    """A mask of the units the seed reaches along directed edges, the seed included."""
    reached = np.zeros(n, dtype=bool)
    reached[seed] = True
    frontier = np.array([seed])
    while frontier.size:
        on = np.zeros(n, dtype=bool)
        on[frontier] = True
        new = dst[on[src]]
        new = np.unique(new[~reached[new]])
        reached[new] = True
        frontier = new
    return reached


def steady(n, src, dst, budget=None, fanout=False, inhibition=False):
    """The plan's dynamics: the seed held at 1; each step a unit becomes half its last value plus its weighted input,
    clipped to [0, 1]; until no unit moves by more than 1e-10 or for 20 000 steps. Returns the activations, whether
    they settled, and the steps taken."""
    outdeg = np.bincount(src, minlength=n)
    indeg = np.bincount(dst, minlength=n)
    if budget is not None:  # each unit's output, budget x activation, divided equally among its targets
        w = budget / outdeg[src]
    elif fanout:  # 0.2 / (1 + ln k) for a source of out-degree k
        w = EDGE / (1.0 + np.log(outdeg[src]))
    else:
        w = np.full(src.size, EDGE)
    a = np.zeros(n)
    a[0] = 1.0
    for step in range(1, SETTLE_STEPS + 1):
        drive = np.bincount(dst, weights=w * a[src], minlength=n)
        if inhibition:  # minus half the mean activation of a unit's inputs
            drive = drive - INHIBITION * np.bincount(dst, weights=a[src], minlength=n) / np.maximum(indeg, 1)
        new = np.clip(DECAY * a + drive, 0.0, 1.0)
        new[0] = 1.0
        moved = np.max(np.abs(new - a))
        a = new
        if moved <= SETTLE_TOL:
            return a, True, step
    return a, False, SETTLE_STEPS


def control_reach(control, n, d, src, dst):
    """The pilot's four controls, as fractions of the other n - 1 units."""
    if control == "neighbours" or (control == "jump" and d < JUMP_AT):
        return np.unique(dst[src == 0]).size / (n - 1)
    return (out_component(n, src, dst).sum() - 1) / (n - 1)


def priming_sums(src, dst, a):
    """Sums and counts of activation over the units one hop and exactly two hops from the seed."""
    one = np.unique(dst[src == 0])
    one = one[one != 0]
    near = np.zeros(a.size, dtype=bool)
    near[one] = True
    near[0] = True
    two = np.unique(dst[np.isin(src, one)])
    two = two[~near[two]]
    return a[one].sum(), one.size, a[two].sum(), two.size


def one_draw(kind, n, d, entropy, priming=False):
    """One eligible draw: (fraction, replaced, settled, steps, priming sums or None)."""
    rng = np.random.default_rng(entropy)
    if kind == "smooth":  # no graph and no seed degree, so never replaced
        p = 1.0 / (1.0 + np.exp(-(d - SMOOTH_MID) / SMOOTH_WIDTH))
        return rng.binomial(n - 1, p) / (n - 1), 0, True, 0, None
    replaced = 0
    while True:
        src, dst = network(n, d, rng)
        if np.any(src == 0):
            break
        replaced += 1
    if kind in CONTROLS:
        return control_reach(kind, n, d, src, dst), replaced, True, 0, None
    a, settled, steps = steady(n, src, dst, **{**BASELINE, **VARIANTS[kind]})
    fraction = np.mean(a[1:] > ACTIVE)
    return fraction, replaced, settled, steps, (priming_sums(src, dst, a) if priming else None)


def group(task):
    """k draws of one kind at one (N, d); the unit of work handed to a process."""
    kind, stage, point, n, d, k, priming = task
    rows = [one_draw(kind, n, d, [SEED, stage, KINDS.index(kind), point, n, i], priming) for i in range(k)]
    return {
        "values": np.array([r[0] for r in rows]),
        "replaced": sum(r[1] for r in rows),
        "unsettled": sum(not r[2] for r in rows),
        "max_steps": max(r[3] for r in rows),
        "priming": np.array([r[4] for r in rows]) if priming else None,
    }


def run(tasks):
    with mp.get_context("fork").Pool(WORKERS) as pool:
        return pool.map(group, tasks, chunksize=1)


# Harness checks: nothing beyond the seed at d = 0; everything on a complete graph of 100, for the reach and the dynamics.
check = np.random.default_rng([SEED, 0])
empty = network(100, 0.0, check)
assert out_component(100, *empty).sum() == 1
a0, settled0, _ = steady(100, *empty, **BASELINE)
assert settled0 and np.all(a0[1:] == 0.0)
complete = np.array([(i, j) for i in range(100) for j in range(100) if i != j]).T
assert out_component(100, complete[0], complete[1]).sum() == 100
a1, settled1, _ = steady(100, complete[0], complete[1], **BASELINE)
assert settled1 and np.all(a1[1:] > ACTIVE)
print("numpy", np.__version__, f"- harness checks passed; generator seeds [{SEED}, stage, kind, point, N, draw];",
      f"levels {LEVEL_CLAUSE:.5f} per clause and {LEVEL_PURE:.5f} for clause (1)'s two; {WORKERS} worker processes")

## Stage 1, Locate

At N = 10 000, 50 draws per degree from 0.25 to 12 by 0.25, for the four controls and then the five variants. D is
where the mean reached fraction first passes 1%, by linear interpolation; the coefficient of variation over draws, the
lead's locator, and the highest mean fraction over the sweep are kept as descriptive readings.

In [ ]:
def locate(kinds):
    tasks = [(kind, 1, i, 10000, float(d), LOCATE_DRAWS, False) for kind in kinds for i, d in enumerate(GRID)]
    out = run(tasks)
    found = {}
    for j, kind in enumerate(kinds):
        part = out[j * GRID.size:(j + 1) * GRID.size]
        means = np.array([g["values"].mean() for g in part])
        with np.errstate(invalid="ignore", divide="ignore"):
            cvs = np.array([g["values"].std(ddof=1) / g["values"].mean() for g in part])
        passing = means >= REACH_LEVEL
        i = int(np.argmax(passing))
        if not passing.any():
            d_hat, first = None, None
        elif i == 0:
            d_hat, first = float(GRID[0]), float(GRID[0])
        else:
            d_hat = float(GRID[i - 1] + (REACH_LEVEL - means[i - 1]) * (GRID[i] - GRID[i - 1]) / (means[i] - means[i - 1]))
            first = float(GRID[i])
        found[kind] = {"d_hat": d_hat, "first": first, "means": means, "cvs": cvs,
                       "cv_peak": float(GRID[int(np.nanargmax(cvs))]) if np.isfinite(cvs).any() else None,
                       "top": float(means.max()), "top_at": float(GRID[int(np.argmax(means))]),
                       "replaced": sum(g["replaced"] for g in part), "unsettled": sum(g["unsettled"] for g in part),
                       "max_steps": max(g["max_steps"] for g in part)}
    return found


def show_located(found):
    print("kind              D       first grid d past 1%   CV peak at d   top mean fraction (at d)   replaced   unsettled of",
          GRID.size * LOCATE_DRAWS)
    for kind, f in found.items():
        shown = "none" if f["d_hat"] is None else f"{f['d_hat']:.3f}"
        first = "none" if f["first"] is None else f"{f['first']:.2f}"
        peak = "none" if f["cv_peak"] is None else f"{f['cv_peak']:.2f}"
        print(f"{kind:16s}  {shown:>6}  {first:>6}                 {peak:>5}          {f['top']:.5f} ({f['top_at']:.2f})"
              f"            {f['replaced']:>5}      {f['unsettled']}")


started = time.perf_counter()
located = locate(CONTROLS)
show_located(located)
print(f"controls located in {time.perf_counter() - started:.0f} s")

## Stage 2, Test

At D / 2 and 2 D (the grid's ends for a kind that never passes 1%), 200 draws at each of N = 100, 1000 and 10 000; the
ratio of the mean reached fraction at N = 100 to that at N = 10 000, with percentile bootstrap intervals at both levels.

In [ ]:
def ratio_interval(a, b, level, entropy):
    """The ratio of means a / b and its percentile bootstrap interval at the given level, a and b resampled independently."""
    rng = np.random.default_rng(entropy)
    ia = rng.integers(0, a.size, (BOOT, a.size))
    ib = rng.integers(0, b.size, (BOOT, b.size))
    with np.errstate(invalid="ignore", divide="ignore"):
        r = a[ia].mean(axis=1) / b[ib].mean(axis=1)
        point = a.mean() / b.mean() if b.mean() > 0 else np.nan
    if not np.all(np.isfinite(r)):
        return point, np.nan, np.nan
    tail = (1 - level) / 2
    return float(point), float(np.quantile(r, tail)), float(np.quantile(r, 1 - tail))


def reading(below, above):
    """The pre-registered reading from the two intervals (low, high); a NaN bound satisfies no margin."""
    finite = below[0] > MARGINS["finite below"]
    extensive = below[1] < MARGINS["extensive below"]
    holding = above[1] < MARGINS["holding above"]
    falling = above[0] > MARGINS["falling above"]
    if finite and holding:
        return "cliff"
    if extensive or falling:
        return "no cliff"
    return "inconclusive"


LEVELS = {"clause": LEVEL_CLAUSE, "pure": LEVEL_PURE}


def test(kinds, found):
    tasks, where = [], []
    for kind in kinds:
        d_hat = found[kind]["d_hat"]
        points = (float(GRID[0]), float(GRID[-1])) if d_hat is None else (d_hat / 2, 2 * d_hat)
        for side, d in enumerate(points):
            for n in SIZES:
                tasks.append((kind, 2, side, n, d, TEST_DRAWS, kind == "pure" and side == 0 and n == 10000))
                where.append((kind, side, n, d))
    out = dict(zip([(k, s, n) for k, s, n, _ in where], run(tasks)))
    results = {}
    for kind in kinds:
        rows = {}
        for side, name in enumerate(("below", "above")):
            d = next(dd for k, s, _, dd in where if k == kind and s == side)
            groups = [out[(kind, side, n)] for n in SIZES]
            means = np.array([g["values"].mean() for g in groups])
            with np.errstate(divide="ignore"):
                slope = float(np.polyfit(np.log(SIZES), np.log(means), 1)[0]) if np.all(means > 0) else np.nan
            intervals = {lv: ratio_interval(groups[0]["values"], groups[2]["values"], LEVELS[lv],
                                            [SEED, 3, KINDS.index(kind), side, j]) for j, lv in enumerate(LEVELS)}
            rows[name] = {"d": d, "means": means, "slope": slope, "intervals": intervals,
                          "replaced": {n: g["replaced"] for n, g in zip(SIZES, groups)},
                          "unsettled": {n: g["unsettled"] for n, g in zip(SIZES, groups)},
                          "max_steps": max(g["max_steps"] for g in groups),
                          "values_10000": groups[2]["values"], "priming": groups[2]["priming"]}
        verdicts = {lv: reading(rows["below"]["intervals"][lv][1:], rows["above"]["intervals"][lv][1:]) for lv in LEVELS}
        results[kind] = {"rows": rows, "verdicts": verdicts}
    return results


def show_tested(results, found):
    for kind, res in results.items():
        d_hat = found[kind]["d_hat"]
        print(f"\n{kind}: D = " + ("none, so the grid's ends stand in" if d_hat is None else f"{d_hat:.3f}"))
        for side in ("below", "above"):
            r = res["rows"][side]
            m = ", ".join(f"{n}: {x:.5f}" for n, x in zip(SIZES, r["means"]))
            iv = "; ".join(f"at {LEVELS[lv]:.5f} [{r['intervals'][lv][1]:.2f}, {r['intervals'][lv][2]:.2f}]" for lv in LEVELS)
            print(f"  {side} (d = {r['d']:.3f}): mean reached fraction by N {m}")
            print(f"    ratio N=100 / N=10000 = {r['intervals']['clause'][0]:.2f}, {iv}; slope {r['slope']:.2f}")
            print(f"    draws replaced {r['replaced']}; unsettled {r['unsettled']}; most steps {r['max_steps']}")
        print("  reads: " + ", ".join(f"{v} at {LEVELS[lv]:.5f}" for lv, v in res["verdicts"].items()))


started = time.perf_counter()
tested = test(CONTROLS, located)
show_tested(tested, located)
controls_ok = all(tested[c]["verdicts"][lv] == MUST_READ[c] for c in CONTROLS for lv in LEVELS)
for c in CONTROLS:
    print(f"{c}: must read {MUST_READ[c]} -> " + ("as it must at both levels" if all(tested[c]["verdicts"][lv] == MUST_READ[c] for lv in LEVELS)
                                                  else "NOT as it must"))
print(f"\ncontrols tested in {time.perf_counter() - started:.0f} s;",
      "the controls read as they must, so the variants are run" if controls_ok else "A CONTROL IS MISREAD: THE RUN IS VOID")

## The Variants

Run only if every control read as it must at both levels. The same two stages, then the 1.1 D reading for each kind
that reads cliff.

In [ ]:
assert controls_ok, "a control is misread: the run is void, and the variants are not run"
started = time.perf_counter()
located.update(locate(list(VARIANTS)))
show_located({k: located[k] for k in VARIANTS})
print(f"variants located in {time.perf_counter() - started:.0f} s")
started = time.perf_counter()
tested.update(test(list(VARIANTS), located))
show_tested({k: tested[k] for k in VARIANTS}, located)
print(f"\nvariants tested in {time.perf_counter() - started:.0f} s")

In [ ]:
# The 1.1 D reading: the mean reached fraction at 1.1 D as a share of that at 2 D, at N = 10 000, for each kind reading cliff.
cliffs = [k for k in KINDS if located[k]["d_hat"] is not None and tested[k]["verdicts"]["clause"] == "cliff"]
started = time.perf_counter()
near = dict(zip(cliffs, run([(k, 4, 2, 10000, 1.1 * located[k]["d_hat"], TEST_DRAWS, False) for k in cliffs])))
share = {}
for k in cliffs:
    at2 = tested[k]["rows"]["above"]["values_10000"].mean()
    share[k] = near[k]["values"].mean() / at2
    print(f"{k:16s} at 1.1 D = {1.1 * located[k]['d_hat']:.3f}: {near[k]['values'].mean():.5f}; at 2 D: {at2:.5f};"
          f" share {share[k]:.3f}; unsettled {near[k]['unsettled']}")
print(f"in {time.perf_counter() - started:.0f} s")

## The Clauses

Each clause as the plan states it. (1) fires if the pure network reads *no cliff* by either of its two intervals at
1 - 0.05 / 12; (2) to (5) fire if that variant reads *cliff* at 1 - 0.05 / 6, unless it never passed 1%, in which case
the clause does not fire and the result says so; (6) fires if the priming ratio's interval at 1 - 0.05 / 6 lies wholly
above 0.5.

In [ ]:
prim = tested["pure"]["rows"]["below"]["priming"]  # per draw: one-hop sum, count, two-hop sum, count
prim_ratio = (prim[:, 2].sum() / prim[:, 3].sum()) / (prim[:, 0].sum() / prim[:, 1].sum())
boot = np.random.default_rng([SEED, 3, KINDS.index("pure"), 9, 0]).integers(0, prim.shape[0], (BOOT, prim.shape[0]))
s = prim[boot].sum(axis=1)
prim_boot = (s[:, 2] / s[:, 3]) / (s[:, 0] / s[:, 1])
tail = (1 - LEVEL_CLAUSE) / 2
prim_lo, prim_hi = np.quantile(prim_boot, tail), np.quantile(prim_boot, 1 - tail)
print(f"priming at the pure network's D / 2 = {tested['pure']['rows']['below']['d']:.3f}, N = 10000, {prim.shape[0]} draws:")
print(f"  one hop: {prim[:, 1].sum():.0f} units, mean activation {prim[:, 0].sum() / prim[:, 1].sum():.4f};"
      f" two hops: {prim[:, 3].sum():.0f} units, mean activation {prim[:, 2].sum() / prim[:, 3].sum():.4f};"
      f" draws with no two-hop unit {int(np.sum(prim[:, 3] == 0))}")
print(f"  ratio two-hop / one-hop {prim_ratio:.3f} [{prim_lo:.3f}, {prim_hi:.3f}] at {LEVEL_CLAUSE:.5f}")

pure = tested["pure"]["rows"]
lo_b, hi_b = pure["below"]["intervals"]["pure"][1:]
lo_a, hi_a = pure["above"]["intervals"]["pure"][1:]
fires = {1: bool(hi_b < MARGINS["extensive below"] or lo_a > MARGINS["falling above"])}
print(f"\n(1) no cliff in the pure network: below [{lo_b:.2f}, {hi_b:.2f}] wholly < 3? {hi_b < 3};"
      f" above [{lo_a:.2f}, {hi_a:.2f}] wholly > 5? {lo_a > 5}; reads {tested['pure']['verdicts']['pure']} -> "
      + ("FIRES" if fires[1] else "does not fire"))
for kind in ("inhibition", "budget c = 0.2", "budget c = 0.6", "fan-out"):
    c, v = CLAUSE[kind], tested[kind]["verdicts"]["clause"]
    spreads = located[kind]["d_hat"] is not None
    fires[c] = bool(spreads and v == "cliff")
    note = "" if spreads else " (never passes 1% at N = 10000: the cure removed the spread, so the clause does not fire)"
    b, a = tested[kind]["rows"]["below"]["intervals"]["clause"], tested[kind]["rows"]["above"]["intervals"]["clause"]
    print(f"({c}) {kind} leaves the cliff: below {b[0]:.2f} [{b[1]:.2f}, {b[2]:.2f}], above {a[0]:.2f} [{a[1]:.2f}, {a[2]:.2f}];"
          f" reads {v} -> " + ("FIRES" if fires[c] else "does not fire") + note)
fires[6] = bool(prim_lo > PRIMING_MARGIN)
print(f"(6) two-hop priming not well below one-hop: [{prim_lo:.3f}, {prim_hi:.3f}] wholly > 0.5? -> "
      + ("FIRES" if fires[6] else "does not fire"))
print("\nclauses that fire:", [c for c in sorted(fires) if fires[c]] or "none")

## The Figure

`docs/figures/c18_cliff_run.svg`: the stage-1 sweep of the five variants at N = 10 000, and the mean reached fraction
against N below and above each one's located threshold, in the pilot's style - one ink, line styles and markers, a fixed
`svg.hashsalt` and no date, so that drawing it again changes no byte. A size at which no draw activated any unit has a
mean of zero and no point on the log axis.

In [ ]:
import hashlib
import io

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["svg.hashsalt"] = "c18-run"
plt.rcParams["svg.fonttype"] = "none"  # text stays text, as in the lab's diagrams: small, and it diffs
FIGURE = "docs/figures/c18_cliff_run.svg"
STYLE = {"pure": ("-", "o"), "budget c = 0.2": (":", "^"), "budget c = 0.6": ("--", "s"), "fan-out": ("-.", "v"),
         "inhibition": ((0, (5, 1, 1, 1, 1, 1)), "D")}


def draw():
    """Three panels: the sweep at N = 10 000, then the fraction against N below and above each D; one ink."""
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.6))
    ax = axes[0]
    for kind in VARIANTS:
        line, _ = STYLE[kind]
        ax.plot(GRID, located[kind]["means"], linestyle=line, color="#1d1d1d", linewidth=1.3, label=kind)
    ax.axhline(REACH_LEVEL, color="#9a9a9a", linewidth=0.8, label="1% (locates D)")
    ax.set_yscale("log")
    ax.set_xlabel("mean degree d")
    ax.set_ylabel("mean reached fraction")
    ax.set_title("Stage 1: the sweep at N = 10 000", fontsize=10)
    ax.grid(True, which="major", color="#e4e4e4", linewidth=0.6)
    ax.legend(fontsize=7, frameon=False)
    for ax, side, title in zip(axes[1:], ("below", "above"), ("Below: at D / 2", "Above: at 2 D")):
        for kind in VARIANTS:
            r = tested[kind]["rows"][side]
            line, marker = STYLE[kind]
            y = np.where(r["means"] > 0, r["means"], np.nan)
            ax.plot(SIZES, y, linestyle=line, marker=marker, color="#1d1d1d", linewidth=1.3, markersize=5,
                    label=f"{kind} (d = {r['d']:.2f})")
        if side == "below":
            ax.plot(SIZES, tested["pure"]["rows"]["below"]["means"][0] * 100 / np.array(SIZES), color="#9a9a9a",
                    linewidth=1, label="1 / N guide")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xticks(SIZES, [str(n) for n in SIZES])
        ax.set_xlabel("N, units")
        ax.set_title(title, fontsize=10)
        ax.grid(True, which="major", color="#e4e4e4", linewidth=0.6)
        ax.legend(fontsize=7, frameon=False)
    fig.suptitle("C18's run: five spreading networks, each located by its 1% crossing and tested below and above it", fontsize=10)
    fig.tight_layout()
    out = io.StringIO()
    fig.savefig(out, format="svg", metadata={"Date": None})
    plt.close(fig)
    return out.getvalue()


first, second = draw(), draw()
assert first == second, "the figure differs between two drawings in one run"
before = open(FIGURE, encoding="utf-8").read() if os.path.exists(FIGURE) else None
status = "new" if before is None else ("unchanged" if before == first else "changed")
with open(FIGURE, "w", encoding="utf-8", newline="\n") as f:
    f.write(first)
print(f"{FIGURE}: {status}, {len(first.encode('utf-8'))} bytes, sha256 {hashlib.sha256(first.encode('utf-8')).hexdigest()}")